# E2 — Global-context and structure-preserving controls
## Is the graph gain local message passing, or simply access to product-level context?

This notebook implements the second pre-JMS diagnostic.

The primary study established that real and semantically permuted graphs outperform a no-edge GNN. E1 then showed that trained models genuinely use cross-node information at inference, but did **not** show a preference for the physically correct held-out topology. E2 asks the next mechanism question:

> Is fixation-like/local graph structure needed, or is generic product-level cross-node context sufficient?

### Conditions

All conditions retain the same product-invariant operation attributes and the same strict 10-fold LOPO protocol.

1. **RealGNN** — recorded physical fixation topology.
2. **NoEdgesGNN** — canonical no-edge intervention.
3. **ShuffledGNN × 5** — the five canonical semantic--topology permutations.
4. **MeanPoolContext** — no edges; every operation receives a global mean summary of the held-out product's component-class node embeddings. Graph-branch degree is zero, so this control carries no physical adjacency or graph-degree information.
5. **CompleteGraphGNN** — all component nodes communicate with all other component nodes using a single dummy edge type. This provides cross-node context without physical locality.
6. **ClassPreservingRewiredGNN × 5** — degree-preserving rewires of the recorded multigraph. Endpoint swaps are restricted to nodes of the same component class, so every fixation record retains its edge type and endpoint-class pair while exact component-to-component adjacency is altered.

### Why the rewiring control is stronger than a free permutation

The canonical semantic--topology permutation preserves the unlabeled graph but changes which semantic nodes occupy structural positions. The new rewiring control instead preserves:

- each node's fixation degree,
- the number and identity of fixation records,
- fixation edge types,
- the ordered endpoint component-class pair of every fixation record,

while changing exact node-to-node connectivity. It therefore asks whether **fine-grained physical adjacency** matters after preserving several lower-order structural statistics.

### Primary E2 question

If `MeanPoolContext` and/or `CompleteGraphGNN` perform similarly to the graph conditions and clearly exceed `NoEdgesGNN`, then the earlier graph gain can be explained largely by access to product-level cross-node context rather than physical locality.

If `RealGNN` or the local graph controls outperform these global-context controls, then locality/graph structure contributes beyond a global product summary.

### Statistical unit

The held-out battery product remains the independent unit (`n=10`). Neural seeds and the five permutation/rewiring realizations are averaged within product. Hits@1 is primary. Exact paired sign-flip tests govern inference; bootstrap intervals describe effect magnitude. E2 is a robustness/mechanism diagnostic, not a replacement for the manuscript's original confirmatory family.


In [ ]:
# 1. Imports and frozen configuration

from pathlib import Path
from collections import Counter, defaultdict
import tarfile, json, math, random, time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

SEEDS = [17, 29, 43]
SHUFFLE_IDS = [1, 2, 3, 4, 5]

EPOCHS = 30
NEGATIVES_PER_STATE = 8
LR = 2e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10

EMBED_DIM = 12
HIDDEN_DIM = 64
GNN_DIM = 32

BASE_SEED = 20261003
GRAPH_SHUFFLE_BASE = 20261119
REWIRE_BASE = 20270117
REWIRE_IDS = [1, 2, 3, 4, 5]

# Keep True for the strongest self-contained E2 experiment.
# Setting False skips the five canonical shuffled conditions and saves time.
RUN_ORIGINAL_PERMUTATIONS = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_E2_CONDITIONS = (
    4  # real, noedge, meanpool, complete
    + len(REWIRE_IDS)
    + (len(SHUFFLE_IDS) if RUN_ORIGINAL_PERMUTATIONS else 0)
)
N_FITS = 10 * len(SEEDS) * N_E2_CONDITIONS

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("Planned E2 neural fits:", N_FITS)

if DEVICE.type != "cuda":
    print("WARNING: GPU is strongly recommended for this notebook.")

In [ ]:
# 2. Find the KIT dataset automatically.
#
# The cell first searches for already-extracted CSVs anywhere under /content.
# If not found, it searches for the uploaded KIT .tar archive and extracts it.

root = Path("/content")

catalogues = list(root.rglob("battery_catalogue.csv"))

if catalogues:
    DATA = catalogues[0].parent
else:
    archives = list(root.rglob("10.35097-emz24pksshndq468.tar"))
    if not archives:
        archives = list(root.rglob("10.35097-*.tar"))

    assert archives, (
        "KIT dataset not found. Upload 10.35097-emz24pksshndq468.tar "
        "to the Colab Files panel, then rerun this cell."
    )

    TAR_PATH = archives[0]
    print("Using archive:", TAR_PATH)

    work = root / "battery_disassembly_dataset_extracted"
    work.mkdir(parents=True, exist_ok=True)

    with tarfile.open(TAR_PATH, "r") as tf:
        tf.extractall(work)

    catalogues = list(work.rglob("battery_catalogue.csv"))
    assert catalogues, "battery_catalogue.csv not found after extraction."
    DATA = catalogues[0].parent

print("Dataset directory:", DATA)

In [ ]:
# 3. Load and validate raw data.

bat = pd.read_csv(DATA / "battery_catalogue.csv")
parts = pd.read_csv(DATA / "parts.csv")
fix = pd.read_csv(DATA / "fixations.csv")
ops = pd.read_csv(DATA / "operations.csv")

print("Batteries:", len(bat))
print("Parts:", len(parts))
print("Fixations:", len(fix))
print("Operations:", len(ops))

display(bat[["id", "name"]])

In [ ]:
# 4. Attach each operation to a battery and reconstruct the 10 reference chains.

battery_names = bat.set_index("id")["name"].to_dict()

fix_battery = fix.set_index("id")["battery_id"].to_dict()
part_battery = parts.set_index("id")["battery_id"].to_dict()
part_class_raw = parts.set_index("id")["class"].to_dict()

fix_info_raw = (
    fix.set_index("id")[["class", "fromPartID", "toPartID"]]
    .to_dict("index")
)

def operation_battery(row):
    if pd.notna(row.fixation_id):
        return int(fix_battery[int(row.fixation_id)])
    return int(part_battery[int(row.part_id)])

ops["battery_id"] = ops.apply(operation_battery, axis=1)
ops_by_id = ops.set_index("id")

chains = {}

for b in sorted(ops["battery_id"].unique()):
    sub = ops[ops["battery_id"] == b]
    ids = set(sub["id"].astype(int))
    next_ids = set(sub["next_op_id"].dropna().astype(int))

    starts = list(ids - next_ids)
    terminals = sub[sub["next_op_id"].isna()]["id"].astype(int).tolist()

    assert len(starts) == 1
    assert len(terminals) == 1

    seq = []
    seen = set()
    cur = int(starts[0])

    while cur in ids and cur not in seen:
        seen.add(cur)
        seq.append(cur)

        nxt = ops_by_id.loc[cur, "next_op_id"]
        if pd.isna(nxt):
            break
        cur = int(nxt)

    assert len(seq) == len(ids), f"Broken chain for battery {b}"
    chains[int(b)] = seq

print({
    battery_names[b]: len(seq)
    for b, seq in chains.items()
})

In [ ]:
# 5. Raw full relational signatures.
#
# These are used only for diagnostics (seen vs unseen motif/signature).
# They are NOT output classes for the neural ranking models.

def raw_full_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)

    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        src_cls = str(part_class_raw[int(fi["fromPartID"])])
        dst_cls = str(part_class_raw[int(fi["toPartID"])])
        return (
            f"{skill}|fix:{fi['class']}|"
            f"{src_cls}->{dst_cls}"
        )

    return (
        f"{skill}|part:"
        f"{part_class_raw[int(r.part_id)]}"
    )

raw_signatures = {
    b: [raw_full_signature(op_id) for op_id in seq]
    for b, seq in chains.items()
}

In [ ]:
# 6. Fold-specific vocabularies.
#
# Vocabularies are learned ONLY from the 9 training batteries.
# Category 0 is <UNK>, so truly unseen test categories do not leak into training.

def make_vocab(values):
    vals = sorted({
        str(x)
        for x in values
        if pd.notna(x)
    })
    return {
        "<UNK>": 0,
        **{v: i + 1 for i, v in enumerate(vals)}
    }

def lookup(vocab, value):
    if pd.isna(value):
        return 0
    return vocab.get(str(value), 0)

def build_fold_vocab(train_batteries):
    return {
        "part": make_vocab(
            parts[
                parts["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "fix": make_vocab(
            fix[
                fix["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "skill": make_vocab(
            ops[
                ops["battery_id"].isin(train_batteries)
            ]["skill_id"]
        ),
    }

In [ ]:
# 7. Convert one battery into product-invariant graph + operation features.
#
# graph_mode:
#   "real"    -> true physical fixation topology
#   "shuffle" -> fixed permutation of topology relative to semantic/operation nodes
#   "noedge"  -> same GNN architecture, but no message-passing edges
#
# Operation-local attributes are identical across all graph modes.
# Therefore only the graph branch is manipulated.

def build_battery_pack(
    battery_id,
    vocab,
    graph_mode="real",
    shuffle_id=0,
):
    assert graph_mode in {"real", "shuffle", "noedge"}

    psub = parts[
        parts["battery_id"] == battery_id
    ].copy()

    fsub = fix[
        fix["battery_id"] == battery_id
    ].copy()

    part_id_to_local = {
        int(pid): i
        for i, pid in enumerate(psub["id"].astype(int))
    }

    n_nodes = len(psub)

    true_degree = np.zeros(n_nodes, dtype=np.float32)

    edge_src = []
    edge_dst = []
    edge_type = []

    for _, r in fsub.iterrows():
        a = part_id_to_local[int(r.fromPartID)]
        b = part_id_to_local[int(r.toPartID)]

        true_degree[a] += 1
        true_degree[b] += 1

        # Undirected message-passing representation.
        edge_src += [a, b]
        edge_dst += [b, a]

        et = lookup(vocab["fix"], r["class"])
        edge_type += [et, et]

    true_degree_norm = (
        true_degree / max(1.0, float(true_degree.max()))
    )

    node_part_class = np.array(
        [
            lookup(vocab["part"], cls)
            for cls in psub["class"]
        ],
        dtype=np.int64,
    )

    true_edge_index = np.array(
        [edge_src, edge_dst],
        dtype=np.int64,
    )

    true_edge_type = np.array(
        edge_type,
        dtype=np.int64,
    )

    if graph_mode == "real":
        graph_edge_index = true_edge_index.copy()
        graph_edge_type = true_edge_type.copy()
        graph_degree_norm = true_degree_norm.copy()

    elif graph_mode == "shuffle":
        # A deterministic, independent permutation for each
        # (shuffle_id, battery). This preserves the unlabeled graph,
        # but reassigns its physical locations to semantic nodes.
        rng_graph = np.random.default_rng(
            GRAPH_SHUFFLE_BASE
            + 10007 * int(shuffle_id)
            + int(battery_id)
        )

        permutation = rng_graph.permutation(n_nodes)

        graph_edge_index = permutation[true_edge_index]
        graph_edge_type = true_edge_type.copy()

        shuffled_degree = np.bincount(
            graph_edge_index.reshape(-1),
            minlength=n_nodes,
        ).astype(np.float32)

        graph_degree_norm = (
            shuffled_degree
            / max(1.0, float(shuffled_degree.max()))
        )

    else:  # noedge
        graph_edge_index = np.empty((2, 0), dtype=np.int64)
        graph_edge_type = np.empty((0,), dtype=np.int64)

        # Do not leak adjacency through the graph branch.
        graph_degree_norm = np.zeros(
            n_nodes,
            dtype=np.float32,
        )

    seq = [int(x) for x in chains[battery_id]]

    categorical = []
    graph_nodes = []
    numeric = []
    signatures = []

    for op_id in seq:
        r = ops_by_id.loc[op_id]

        skill_idx = lookup(
            vocab["skill"],
            r.skill_id,
        )

        if pd.notna(r.fixation_id):
            fi = fix_info_raw[int(r.fixation_id)]

            src_pid = int(fi["fromPartID"])
            dst_pid = int(fi["toPartID"])

            src_node = part_id_to_local[src_pid]
            dst_node = part_id_to_local[dst_pid]

            categorical.append([
                skill_idx,
                0,
                lookup(vocab["fix"], fi["class"]),
                lookup(vocab["part"], part_class_raw[src_pid]),
                lookup(vocab["part"], part_class_raw[dst_pid]),
                0,
            ])

            graph_nodes.append([
                src_node,
                dst_node,
                -1,
            ])

            # IMPORTANT:
            # identical operation-local features in every graph condition.
            numeric.append([
                true_degree_norm[src_node],
                true_degree_norm[dst_node],
                0.0,
            ])

        else:
            pid = int(r.part_id)
            node = part_id_to_local[pid]

            categorical.append([
                skill_idx,
                1,
                0,
                0,
                0,
                lookup(vocab["part"], part_class_raw[pid]),
            ])

            graph_nodes.append([
                -1,
                -1,
                node,
            ])

            numeric.append([
                0.0,
                0.0,
                true_degree_norm[node],
            ])

        signatures.append(raw_full_signature(op_id))

    return {
        "battery_id": int(battery_id),
        "battery_name": battery_names[int(battery_id)],
        "node_part_class": node_part_class,
        "node_degree": graph_degree_norm.astype(np.float32),
        "edge_index": graph_edge_index,
        "edge_type": graph_edge_type,
        "op_categorical": np.asarray(categorical, dtype=np.int64),
        "op_graph_nodes": np.asarray(graph_nodes, dtype=np.int64),
        "op_numeric": np.asarray(numeric, dtype=np.float32),
        "full_signatures": signatures,
        "n_ops": len(seq),
        "graph_mode": graph_mode,
        "shuffle_id": int(shuffle_id),
    }

In [ ]:
# 8. Neural models.

class BaseOperationRanker(nn.Module):
    """
    Product-invariant MLP operation ranker.

    It never receives battery/product IDs or raw entity IDs.
    """

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.skill_emb = nn.Embedding(n_skill, emb_dim)
        self.kind_emb = nn.Embedding(2, 4)
        self.fix_emb = nn.Embedding(n_fix, emb_dim)
        self.part_emb = nn.Embedding(n_part, emb_dim)

        op_input_dim = (
            emb_dim      # skill
            + 4          # operation kind
            + emb_dim    # fixation class
            + emb_dim    # source part class
            + emb_dim    # target part class
            + emb_dim    # removed part class
            + 3          # normalized local degrees
        )

        self.op_mlp = nn.Sequential(
            nn.Linear(op_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # current, candidate, abs difference, product, progress
        self.score_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def base_operation_encoding(self, pack):
        cats = torch.as_tensor(
            pack["op_categorical"],
            dtype=torch.long,
            device=DEVICE,
        )

        nums = torch.as_tensor(
            pack["op_numeric"],
            dtype=torch.float32,
            device=DEVICE,
        )

        skill = cats[:, 0]
        kind = cats[:, 1]
        fix_class = cats[:, 2]
        src_class = cats[:, 3]
        dst_class = cats[:, 4]
        part_class = cats[:, 5]

        x = torch.cat([
            self.skill_emb(skill),
            self.kind_emb(kind),
            self.fix_emb(fix_class),
            self.part_emb(src_class),
            self.part_emb(dst_class),
            self.part_emb(part_class),
            nums,
        ], dim=-1)

        return self.op_mlp(x)

    def encode_operations(self, pack):
        return self.base_operation_encoding(pack)

    def score_pairs(self, current_repr, candidate_repr, progress):
        x = torch.cat([
            current_repr,
            candidate_repr,
            torch.abs(current_repr - candidate_repr),
            current_repr * candidate_repr,
            progress,
        ], dim=-1)

        return self.score_mlp(x).squeeze(-1)


class GraphEncoder(nn.Module):
    """
    Two-layer product-invariant message-passing network.
    """

    def __init__(
        self,
        n_part,
        n_fix,
        part_emb_dim=EMBED_DIM,
        edge_emb_dim=8,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.part_emb = nn.Embedding(n_part, part_emb_dim)
        self.edge_emb = nn.Embedding(n_fix, edge_emb_dim)

        self.input_proj = nn.Linear(
            part_emb_dim + 1,
            graph_dim,
        )

        self.message_layers = nn.ModuleList([
            nn.Linear(
                graph_dim + edge_emb_dim,
                graph_dim,
            )
            for _ in range(2)
        ])

        self.self_layers = nn.ModuleList([
            nn.Linear(graph_dim, graph_dim)
            for _ in range(2)
        ])

        self.dropout = dropout

    def forward(self, pack):
        node_class = torch.as_tensor(
            pack["node_part_class"],
            dtype=torch.long,
            device=DEVICE,
        )

        degree = torch.as_tensor(
            pack["node_degree"],
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(-1)

        h = F.relu(
            self.input_proj(
                torch.cat([
                    self.part_emb(node_class),
                    degree,
                ], dim=-1)
            )
        )

        edge_index = torch.as_tensor(
            pack["edge_index"],
            dtype=torch.long,
            device=DEVICE,
        )

        edge_type = torch.as_tensor(
            pack["edge_type"],
            dtype=torch.long,
            device=DEVICE,
        )

        src = edge_index[0]
        dst = edge_index[1]

        edge_embedding = self.edge_emb(edge_type)

        for message_layer, self_layer in zip(
            self.message_layers,
            self.self_layers,
        ):
            msg = message_layer(
                torch.cat([
                    h[src],
                    edge_embedding,
                ], dim=-1)
            )

            agg = torch.zeros(
                (h.shape[0], h.shape[1]),
                dtype=h.dtype,
                device=DEVICE,
            )

            agg.index_add_(0, dst, msg)

            counts = torch.zeros(
                (h.shape[0], 1),
                dtype=h.dtype,
                device=DEVICE,
            )

            counts.index_add_(
                0,
                dst,
                torch.ones(
                    (dst.numel(), 1),
                    dtype=h.dtype,
                    device=DEVICE,
                ),
            )

            agg = agg / counts.clamp_min(1.0)

            h = F.relu(
                self_layer(h) + agg
            )

            h = F.dropout(
                h,
                p=self.dropout,
                training=self.training,
            )

        return h


class GNNOperationRanker(BaseOperationRanker):

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__(
            n_skill=n_skill,
            n_fix=n_fix,
            n_part=n_part,
            emb_dim=emb_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.graph_encoder = GraphEncoder(
            n_part=n_part,
            n_fix=n_fix,
            part_emb_dim=emb_dim,
            graph_dim=graph_dim,
            dropout=dropout,
        )

        # src embedding, dst embedding, absolute difference
        self.structural_projection = nn.Sequential(
            nn.Linear(graph_dim * 3, hidden_dim),
            nn.ReLU(),
        )

        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
        )

    def encode_operations(self, pack):
        base = self.base_operation_encoding(pack)
        node_h = self.graph_encoder(pack)

        op_nodes = torch.as_tensor(
            pack["op_graph_nodes"],
            dtype=torch.long,
            device=DEVICE,
        )

        n_ops = op_nodes.shape[0]

        a = torch.zeros(
            (n_ops, node_h.shape[1]),
            device=DEVICE,
        )

        b = torch.zeros_like(a)

        fixation_mask = op_nodes[:, 0] >= 0

        if fixation_mask.any():
            a[fixation_mask] = node_h[
                op_nodes[fixation_mask, 0]
            ]

            b[fixation_mask] = node_h[
                op_nodes[fixation_mask, 1]
            ]

        part_mask = op_nodes[:, 2] >= 0

        if part_mask.any():
            a[part_mask] = node_h[
                op_nodes[part_mask, 2]
            ]

        structural = self.structural_projection(
            torch.cat([
                a,
                b,
                torch.abs(a - b),
            ], dim=-1)
        )

        return self.combine(
            torch.cat([
                base,
                structural,
            ], dim=-1)
        )

In [ ]:
# 9. Reproducibility + pairwise-ranking training

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_training_pairs(n_ops, k, rng):
    current, positive, negative, progress = [], [], [], []

    for t in range(n_ops - 2):
        pool = np.arange(
            t + 2,
            n_ops,
            dtype=np.int64,
        )

        neg = rng.choice(
            pool,
            size=k,
            replace=(len(pool) < k),
        )

        current.append(t)
        positive.append(t + 1)
        negative.append(neg)
        progress.append(
            t / max(1, n_ops - 1)
        )

    return (
        np.asarray(current, dtype=np.int64),
        np.asarray(positive, dtype=np.int64),
        np.stack(negative),
        np.asarray(progress, dtype=np.float32),
    )


def build_gnn(vocab):
    model = GNNOperationRanker(
        n_skill=len(vocab["skill"]),
        n_fix=len(vocab["fix"]),
        n_part=len(vocab["part"]),
    )
    return model.to(DEVICE)


def train_one_gnn_condition(
    train_batteries,
    test_battery,
    seed,
    graph_mode,
    shuffle_id=0,
):
    set_seed(seed)

    vocab = build_fold_vocab(train_batteries)

    packs = {
        b: build_battery_pack(
            b,
            vocab,
            graph_mode=graph_mode,
            shuffle_id=shuffle_id,
        )
        for b in train_batteries + [test_battery]
    }

    model = build_gnn(vocab)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    # Using the same training seed across paired graph conditions
    # keeps initialization, training order, and negative sampling paired.
    rng = np.random.default_rng(seed)

    for epoch in range(EPOCHS):
        order = list(train_batteries)
        rng.shuffle(order)

        model.train()

        for b in order:
            pack = packs[b]

            cur, pos, neg, prog = make_training_pairs(
                pack["n_ops"],
                NEGATIVES_PER_STATE,
                rng,
            )

            z = model.encode_operations(pack)

            cur_t = torch.as_tensor(
                cur,
                dtype=torch.long,
                device=DEVICE,
            )

            pos_t = torch.as_tensor(
                pos,
                dtype=torch.long,
                device=DEVICE,
            )

            neg_t = torch.as_tensor(
                neg,
                dtype=torch.long,
                device=DEVICE,
            )

            prog_t = torch.as_tensor(
                prog,
                dtype=torch.float32,
                device=DEVICE,
            ).unsqueeze(-1)

            cur_z = z[cur_t]
            pos_z = z[pos_t]

            pos_score = model.score_pairs(
                cur_z,
                pos_z,
                prog_t,
            )

            k = neg_t.shape[1]

            cur_rep = (
                cur_z[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, cur_z.shape[1])
            )

            neg_z = z[neg_t.reshape(-1)]

            prog_rep = (
                prog_t[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, 1)
            )

            neg_score = model.score_pairs(
                cur_rep,
                neg_z,
                prog_rep,
            ).view(len(cur), k)

            loss = F.softplus(
                -(pos_score[:, None] - neg_score)
            ).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model, packs[test_battery], vocab

In [ ]:
# 10. Tie-aware ranking metrics.

def expected_tie_metrics(scores, target_position=0, tol=1e-8):
    """
    Scores correspond to candidate operations.

    Returns the expected Hits@1, Hits@3, and reciprocal rank
    under random ordering within exact/near score ties.
    """

    scores = np.asarray(scores)
    target_score = scores[target_position]

    greater = int(
        np.sum(scores > target_score + tol)
    )

    equal = int(
        np.sum(
            np.abs(scores - target_score) <= tol
        )
    )

    equal = max(equal, 1)

    hits1 = (
        max(
            0,
            min(equal, 1 - greater),
        )
        / equal
        if greater < 1
        else 0.0
    )

    hits3 = (
        max(
            0,
            min(equal, 3 - greater),
        )
        / equal
        if greater < 3
        else 0.0
    )

    ranks = np.arange(
        greater + 1,
        greater + equal + 1,
        dtype=float,
    )

    mrr = float(
        np.mean(1.0 / ranks)
    )

    return hits1, hits3, mrr

In [ ]:
# 11. Evaluate one neural model on an unseen battery.

def training_signature_set(train_batteries):
    s = set()

    for b in train_batteries:
        s.update(raw_signatures[b])

    return s


def evaluate_neural_model(
    model,
    test_pack,
    train_batteries,
):
    model.eval()

    seen_signatures = training_signature_set(
        train_batteries
    )

    rows = []

    with torch.no_grad():
        operation_repr = model.encode_operations(
            test_pack
        )

        n_ops = test_pack["n_ops"]

        for t in range(n_ops - 1):
            candidate_indices = np.arange(
                t + 1,
                n_ops,
                dtype=np.int64,
            )

            candidate_t = torch.as_tensor(
                candidate_indices,
                dtype=torch.long,
                device=DEVICE,
            )

            current_repr = (
                operation_repr[t]
                .unsqueeze(0)
                .expand(
                    len(candidate_indices),
                    -1,
                )
            )

            candidate_repr = operation_repr[
                candidate_t
            ]

            progress = torch.full(
                (len(candidate_indices), 1),
                t / max(1, n_ops - 1),
                dtype=torch.float32,
                device=DEVICE,
            )

            scores = model.score_pairs(
                current_repr,
                candidate_repr,
                progress,
            ).cpu().numpy()

            # candidate_indices[0] is the reference next operation.
            h1, h3, mrr = expected_tie_metrics(
                scores,
                target_position=0,
            )

            target_signature = (
                test_pack["full_signatures"][t + 1]
            )

            rows.append({
                "position": t,
                "hits1": h1,
                "hits3": h3,
                "mrr": mrr,
                "signature_seen_in_training":
                    target_signature in seen_signatures,
            })

    detail = pd.DataFrame(rows)

    metrics = {
        "hits1": float(detail["hits1"].mean()),
        "hits3": float(detail["hits3"].mean()),
        "mrr": float(detail["mrr"].mean()),
        "seen_signature_rate": float(
            detail["signature_seen_in_training"].mean()
        ),
    }

    seen = detail[
        detail["signature_seen_in_training"]
    ]

    unseen = detail[
        ~detail["signature_seen_in_training"]
    ]

    for label, subset in [
        ("seen", seen),
        ("unseen", unseen),
    ]:
        if len(subset) > 0:
            metrics[f"{label}_hits1"] = float(
                subset["hits1"].mean()
            )

            metrics[f"{label}_hits3"] = float(
                subset["hits3"].mean()
            )

            metrics[f"{label}_mrr"] = float(
                subset["mrr"].mean()
            )
        else:
            metrics[f"{label}_hits1"] = np.nan
            metrics[f"{label}_hits3"] = np.nan
            metrics[f"{label}_mrr"] = np.nan

    return metrics, detail

In [ ]:
# 12. Exact-capacity validity helper

def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

## 13. E2 graph builders

`build_battery_pack` remains unchanged for the canonical real, shuffled, and no-edge conditions.

The helper below creates the three new E2 interventions:

- `meanpool`: no cross-node edges and zero graph-branch degree;
- `complete`: directed all-to-all edges excluding self-loops, one dummy edge type, constant graph-branch degree;
- `rewire`: component-class-preserving endpoint swaps on fixation records.

Operation-level semantic inputs and operation-local normalized physical-degree features remain exactly the same as in the primary experiment.


In [ ]:
# 13. New E2 graph-pack builders

from collections import defaultdict

def _part_local_maps(battery_id):
    psub = parts[parts["battery_id"] == battery_id].copy()
    part_ids = psub["id"].astype(int).tolist()
    part_id_to_local = {pid: i for i, pid in enumerate(part_ids)}
    local_raw_class = {
        i: str(cls)
        for i, cls in enumerate(psub["class"].tolist())
    }
    return psub, part_id_to_local, local_raw_class


def _connected_components_count(n_nodes, undirected_edges):
    adj = [[] for _ in range(n_nodes)]
    for u, v in undirected_edges:
        if u == v:
            continue
        adj[u].append(v)
        adj[v].append(u)

    seen = set()
    n_comp = 0
    for start in range(n_nodes):
        if start in seen:
            continue
        n_comp += 1
        stack = [start]
        seen.add(start)
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if v not in seen:
                    seen.add(v)
                    stack.append(v)
    return n_comp


def class_preserving_rewire_records(
    battery_id,
    vocab,
    rewire_id,
    swaps_per_edge=20,
):
    """
    Degree-preserving endpoint swaps on fixation records.

    We swap two selected endpoints only when their raw component classes are
    identical. Edge type stays attached to each fixation record.

    Preserved exactly:
      - node fixation degree,
      - number of fixation records,
      - edge-type multiset,
      - ordered endpoint-class pair of each fixation record.

    Parallel edges are allowed because the source graph is a multigraph.
    Self-loops are rejected.
    """
    psub, part_id_to_local, local_raw_class = _part_local_maps(battery_id)
    fsub = fix[fix["battery_id"] == battery_id].copy()

    records = []
    for _, r in fsub.iterrows():
        u = part_id_to_local[int(r.fromPartID)]
        v = part_id_to_local[int(r.toPartID)]
        et = lookup(vocab["fix"], r["class"])
        records.append([u, v, et])

    original = [x.copy() for x in records]
    n_nodes = len(psub)
    n_edges = len(records)

    original_degree = np.zeros(n_nodes, dtype=np.int64)
    for u, v, _ in records:
        original_degree[u] += 1
        original_degree[v] += 1

    original_components = _connected_components_count(
        n_nodes,
        [(u, v) for u, v, _ in records],
    )

    rng = np.random.default_rng(
        REWIRE_BASE + 10007 * int(rewire_id) + int(battery_id)
    )

    endpoint_refs = defaultdict(list)
    for ei, (u, v, _) in enumerate(records):
        endpoint_refs[local_raw_class[u]].append((ei, 0))
        endpoint_refs[local_raw_class[v]].append((ei, 1))

    eligible_classes = [
        cls for cls, refs in endpoint_refs.items()
        if len(refs) >= 2
    ]
    assert eligible_classes, f"No eligible class-preserving swaps for battery {battery_id}"

    target = max(1, swaps_per_edge * n_edges)
    max_attempts = max(target * 50, 10000)
    accepted = 0
    attempts = 0

    while accepted < target and attempts < max_attempts:
        attempts += 1
        cls = eligible_classes[int(rng.integers(0, len(eligible_classes)))]
        refs = endpoint_refs[cls]

        idx = rng.choice(len(refs), size=2, replace=False)
        (e1, p1), (e2, p2) = refs[int(idx[0])], refs[int(idx[1])]

        if e1 == e2:
            continue

        n1 = records[e1][p1]
        n2 = records[e2][p2]

        if n1 == n2:
            continue

        other1 = records[e1][1 - p1]
        other2 = records[e2][1 - p2]

        # Reject self-loops after swapping.
        if n2 == other1 or n1 == other2:
            continue

        # Because n1 and n2 have the same raw class, each edge record keeps
        # its original ordered endpoint-class pair.
        records[e1][p1] = n2
        records[e2][p2] = n1
        accepted += 1

    new_degree = np.zeros(n_nodes, dtype=np.int64)
    for u, v, _ in records:
        new_degree[u] += 1
        new_degree[v] += 1

    assert np.array_equal(original_degree, new_degree)

    # Every fixation record must preserve its endpoint classes and edge type.
    for old, new in zip(original, records):
        ou, ov, ot = old
        nu, nv, nt = new
        assert ot == nt
        assert local_raw_class[ou] == local_raw_class[nu]
        assert local_raw_class[ov] == local_raw_class[nv]

    changed_records = sum(
        1 for old, new in zip(original, records)
        if old[0] != new[0] or old[1] != new[1]
    )
    changed_fraction = changed_records / max(1, n_edges)

    new_components = _connected_components_count(
        n_nodes,
        [(u, v) for u, v, _ in records],
    )

    return {
        "records": records,
        "accepted_swaps": int(accepted),
        "attempts": int(attempts),
        "changed_fraction": float(changed_fraction),
        "original_components": int(original_components),
        "rewired_components": int(new_components),
    }


def build_e2_pack(
    battery_id,
    vocab,
    condition,
    realization_id=0,
):
    """
    Build one battery pack for any E2 condition.
    """
    if condition == "real":
        return build_battery_pack(
            battery_id, vocab, graph_mode="real", shuffle_id=0
        )

    if condition == "noedge":
        return build_battery_pack(
            battery_id, vocab, graph_mode="noedge", shuffle_id=0
        )

    if condition == "shuffle":
        return build_battery_pack(
            battery_id, vocab, graph_mode="shuffle", shuffle_id=realization_id
        )

    # Start from the real pack so operation inputs remain byte-for-byte
    # identical to the primary experiment.
    pack = build_battery_pack(
        battery_id, vocab, graph_mode="real", shuffle_id=0
    )

    n_nodes = len(pack["node_part_class"])

    if condition == "meanpool":
        pack["edge_index"] = np.empty((2, 0), dtype=np.int64)
        pack["edge_type"] = np.empty((0,), dtype=np.int64)
        pack["node_degree"] = np.zeros(n_nodes, dtype=np.float32)
        pack["graph_mode"] = "meanpool"
        pack["realization_id"] = 0
        return pack

    if condition == "complete":
        src = []
        dst = []
        for u in range(n_nodes):
            for v in range(n_nodes):
                if u != v:
                    src.append(u)
                    dst.append(v)

        pack["edge_index"] = np.asarray([src, dst], dtype=np.int64)
        # Dummy/unknown edge type only; no physical fixation type information.
        pack["edge_type"] = np.zeros(len(src), dtype=np.int64)
        # Complete-graph normalized degree is constant for every node.
        pack["node_degree"] = np.ones(n_nodes, dtype=np.float32)
        pack["graph_mode"] = "complete"
        pack["realization_id"] = 0
        return pack

    if condition == "rewire":
        info = class_preserving_rewire_records(
            battery_id=battery_id,
            vocab=vocab,
            rewire_id=realization_id,
        )

        edge_src = []
        edge_dst = []
        edge_type = []

        for u, v, et in info["records"]:
            edge_src += [u, v]
            edge_dst += [v, u]
            edge_type += [et, et]

        pack["edge_index"] = np.asarray(
            [edge_src, edge_dst],
            dtype=np.int64,
        )
        pack["edge_type"] = np.asarray(edge_type, dtype=np.int64)

        # Degree sequence is preserved exactly, so the real-pack degree input
        # remains valid and identical node by node.
        pack["graph_mode"] = "rewire"
        pack["realization_id"] = int(realization_id)
        pack["rewire_accepted_swaps"] = info["accepted_swaps"]
        pack["rewire_attempts"] = info["attempts"]
        pack["rewire_changed_fraction"] = info["changed_fraction"]
        pack["rewire_original_components"] = info["original_components"]
        pack["rewire_components"] = info["rewired_components"]
        return pack

    raise ValueError(f"Unknown E2 condition: {condition}")


## 14. Mean-pool product-context model

This control deliberately removes adjacency. It uses component-class embeddings to form a single mean product-context vector and supplies that context to every operation representation.

It does **not** use graph-branch physical degree. Operation-local features remain unchanged, matching the primary representation.

If this simpler global-context model approaches the graph conditions, the earlier gain cannot be attributed specifically to local message passing.


In [ ]:
# 14. Mean-pool product-context ranker

class MeanPoolContextRanker(BaseOperationRanker):
    """
    Operation ranker with a global product-composition context but no edges.
    """

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__(
            n_skill=n_skill,
            n_fix=n_fix,
            n_part=n_part,
            emb_dim=emb_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        # Reuse the BaseOperationRanker's part embedding table so the global
        # context does not introduce a second semantic embedding vocabulary.
        self.context_input = nn.Linear(emb_dim + 1, graph_dim)

        self.context_mlp = nn.Sequential(
            nn.Linear(graph_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.context_combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
        )

    def encode_operations(self, pack):
        base = self.base_operation_encoding(pack)

        node_class = torch.as_tensor(
            pack["node_part_class"],
            dtype=torch.long,
            device=DEVICE,
        )

        # E2 mean-pool explicitly carries no graph-degree information.
        zero_degree = torch.zeros(
            (len(node_class), 1),
            dtype=torch.float32,
            device=DEVICE,
        )

        node_h = F.relu(
            self.context_input(
                torch.cat(
                    [self.part_emb(node_class), zero_degree],
                    dim=-1,
                )
            )
        )

        product_context = node_h.mean(dim=0, keepdim=True)
        product_context = self.context_mlp(product_context)
        product_context = product_context.expand(base.shape[0], -1)

        return self.context_combine(
            torch.cat([base, product_context], dim=-1)
        )


def build_e2_model(vocab, condition):
    if condition == "meanpool":
        model = MeanPoolContextRanker(
            n_skill=len(vocab["skill"]),
            n_fix=len(vocab["fix"]),
            n_part=len(vocab["part"]),
        )
    else:
        model = GNNOperationRanker(
            n_skill=len(vocab["skill"]),
            n_fix=len(vocab["fix"]),
            n_part=len(vocab["part"]),
        )

    return model.to(DEVICE)


## 15. Generic E2 training routine and sanity checks

The optimizer, training seeds, product order, negative sampling, epochs, and pairwise ranking objective are unchanged from the canonical experiment.


In [ ]:
# 15. Generic E2 training and graph-control diagnostics

def train_one_e2_condition(
    train_batteries,
    test_battery,
    seed,
    condition,
    realization_id=0,
):
    set_seed(seed)
    vocab = build_fold_vocab(train_batteries)

    packs = {
        b: build_e2_pack(
            b,
            vocab,
            condition=condition,
            realization_id=realization_id,
        )
        for b in train_batteries + [test_battery]
    }

    model = build_e2_model(vocab, condition)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    rng = np.random.default_rng(seed)

    for epoch in range(EPOCHS):
        order = list(train_batteries)
        rng.shuffle(order)
        model.train()

        for b in order:
            pack = packs[b]

            cur, pos, neg, prog = make_training_pairs(
                pack["n_ops"],
                NEGATIVES_PER_STATE,
                rng,
            )

            z = model.encode_operations(pack)

            cur_t = torch.as_tensor(
                cur, dtype=torch.long, device=DEVICE
            )
            pos_t = torch.as_tensor(
                pos, dtype=torch.long, device=DEVICE
            )
            neg_t = torch.as_tensor(
                neg, dtype=torch.long, device=DEVICE
            )
            prog_t = torch.as_tensor(
                prog, dtype=torch.float32, device=DEVICE
            ).unsqueeze(-1)

            cur_z = z[cur_t]
            pos_z = z[pos_t]

            pos_score = model.score_pairs(
                cur_z,
                pos_z,
                prog_t,
            )

            k = neg_t.shape[1]

            cur_rep = (
                cur_z[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, cur_z.shape[1])
            )
            neg_z = z[neg_t.reshape(-1)]
            prog_rep = (
                prog_t[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, 1)
            )

            neg_score = model.score_pairs(
                cur_rep,
                neg_z,
                prog_rep,
            ).reshape(len(cur_z), k)

            loss = F.softplus(
                -(pos_score[:, None] - neg_score)
            ).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    test_pack = packs[test_battery]
    metrics, detail = evaluate_neural_model(
        model,
        test_pack,
        train_batteries,
    )

    return model, metrics, detail, vocab, test_pack


# ---------- structural sanity check on one fold ----------

TEST_BATTERY = sorted(chains)[0]
train_batteries = [b for b in sorted(chains) if b != TEST_BATTERY]
vocab = build_fold_vocab(train_batteries)

real_pack = build_e2_pack(TEST_BATTERY, vocab, "real")
noedge_pack = build_e2_pack(TEST_BATTERY, vocab, "noedge")
meanpool_pack = build_e2_pack(TEST_BATTERY, vocab, "meanpool")
complete_pack = build_e2_pack(TEST_BATTERY, vocab, "complete")
rewire_pack = build_e2_pack(TEST_BATTERY, vocab, "rewire", 1)

# Operation-level semantic inputs remain identical across interventions.
for p in [noedge_pack, meanpool_pack, complete_pack, rewire_pack]:
    assert np.array_equal(real_pack["op_categorical"], p["op_categorical"])
    assert np.array_equal(real_pack["op_graph_nodes"], p["op_graph_nodes"])
    assert np.allclose(real_pack["op_numeric"], p["op_numeric"])

assert noedge_pack["edge_index"].shape[1] == 0
assert meanpool_pack["edge_index"].shape[1] == 0
assert np.allclose(meanpool_pack["node_degree"], 0.0)
assert np.allclose(complete_pack["node_degree"], 1.0)

# Rewiring must preserve node degree exactly.
assert np.allclose(real_pack["node_degree"], rewire_pack["node_degree"])

print("Sanity checks passed for:", battery_names[TEST_BATTERY])
print("Real directed edges:", real_pack["edge_index"].shape[1])
print("Complete directed edges:", complete_pack["edge_index"].shape[1])
print("Rewire changed fixation-record fraction:",
      rewire_pack["rewire_changed_fraction"])
print("Rewire connected components: original =",
      rewire_pack["rewire_original_components"],
      ", rewired =", rewire_pack["rewire_components"])

# Parameter-count context
for cond in ["real", "meanpool", "complete"]:
    model = build_e2_model(vocab, cond)
    print(cond, "trainable parameters:", count_trainable_parameters(model))
    del model


## 16. Run the full E2 experiment

Default configuration runs **420 neural fits**:

- RealGNN: 30
- NoEdgesGNN: 30
- MeanPoolContext: 30
- CompleteGraphGNN: 30
- five canonical semantic--topology permutations: 150
- five class-preserving rewires: 150

The run is resumable through a partial CSV. Because complete graphs are denser than the physical graphs, this experiment is best run on an L4 or similar GPU.


In [ ]:
# 16. Full E2 experiment

import shutil
import itertools

OUT = Path("/content/battery_e2_global_context_lopo")
OUT.mkdir(parents=True, exist_ok=True)
PARTIAL = OUT / "e2_partial.csv"

if PARTIAL.exists():
    e2_results = pd.read_csv(PARTIAL)
    print("Resuming from", len(e2_results), "completed fits.")
else:
    e2_results = pd.DataFrame()

battery_ids = sorted(chains)

conditions = [
    ("RealGNN", "real", 0),
    ("NoEdgesGNN", "noedge", 0),
    ("MeanPoolContext", "meanpool", 0),
    ("CompleteGraphGNN", "complete", 0),
]

if RUN_ORIGINAL_PERMUTATIONS:
    conditions += [
        ("ShuffledGNN", "shuffle", sid)
        for sid in SHUFFLE_IDS
    ]

conditions += [
    ("ClassPreservingRewiredGNN", "rewire", rid)
    for rid in REWIRE_IDS
]

def fit_done(df, test_battery, seed, condition, realization_id):
    if len(df) == 0:
        return False
    mask = (
        (df["battery_id"] == int(test_battery))
        & (df["seed"] == int(seed))
        & (df["condition"] == str(condition))
        & (df["realization_id"] == int(realization_id))
    )
    return bool(mask.any())


start = time.time()

for fold_idx, test_battery in enumerate(battery_ids, start=1):
    train_batteries = [b for b in battery_ids if b != test_battery]

    print(
        f"\nFold {fold_idx}/10 — held out: "
        f"{battery_names[test_battery]}"
    )

    for label, condition, realization_id in conditions:
        for seed in SEEDS:
            if fit_done(
                e2_results,
                test_battery,
                seed,
                condition,
                realization_id,
            ):
                print(
                    f"  skip {condition} r={realization_id} seed={seed}"
                )
                continue

            model, metrics, _, vocab, test_pack = train_one_e2_condition(
                train_batteries=train_batteries,
                test_battery=test_battery,
                seed=seed,
                condition=condition,
                realization_id=realization_id,
            )

            row = {
                "battery_id": int(test_battery),
                "battery": battery_names[test_battery],
                "model": label,
                "condition": condition,
                "realization_id": int(realization_id),
                "seed": int(seed),
                "hits1": float(metrics["hits1"]),
                "hits3": float(metrics["hits3"]),
                "mrr": float(metrics["mrr"]),
                "seen_signature_rate": float(metrics["seen_signature_rate"]),
                "parameter_count": int(count_trainable_parameters(model)),
                "n_nodes": int(len(test_pack["node_part_class"])),
                "n_directed_edges": int(test_pack["edge_index"].shape[1]),
                "rewire_changed_fraction": float(
                    test_pack.get("rewire_changed_fraction", np.nan)
                ),
                "rewire_original_components": float(
                    test_pack.get("rewire_original_components", np.nan)
                ),
                "rewire_components": float(
                    test_pack.get("rewire_components", np.nan)
                ),
            }

            e2_results = pd.concat(
                [e2_results, pd.DataFrame([row])],
                ignore_index=True,
            )
            e2_results.to_csv(PARTIAL, index=False)

            print(
                f"  done {condition} r={realization_id} seed={seed}: "
                f"H1={metrics['hits1']:.3f}"
            )

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

elapsed = (time.time() - start) / 60
expected = 10 * len(SEEDS) * len(conditions)

print(
    f"\nCompleted/resumed to {len(e2_results)} fits "
    f"(expected {expected}) in {elapsed:.1f} min."
)


## 17. Aggregate realizations and neural seeds within held-out product

For shuffled and rewired conditions, the five realizations are averaged within each `(product, seed)` first. Then the three neural seeds are averaged. This yields one observation per held-out product for every named condition.


In [ ]:
# 17. Product-level aggregation

expected = 10 * len(SEEDS) * len(conditions)
assert len(e2_results) == expected, (
    f"E2 incomplete: {len(e2_results)} fits present; {expected} expected."
)

# Average the five graph realizations within seed when applicable.
seed_rows = []

for (battery_id, battery, seed), grp in e2_results.groupby(
    ["battery_id", "battery", "seed"]
):
    for cond_name in ["real", "noedge", "meanpool", "complete"]:
        g = grp[grp["condition"] == cond_name]
        assert len(g) == 1
        seed_rows.append({
            "battery_id": battery_id,
            "battery": battery,
            "seed": seed,
            "condition_summary": cond_name,
            "hits1": g["hits1"].iloc[0],
            "hits3": g["hits3"].iloc[0],
            "mrr": g["mrr"].iloc[0],
            "parameter_count": g["parameter_count"].iloc[0],
            "realization_sd_hits1": 0.0,
        })

    if RUN_ORIGINAL_PERMUTATIONS:
        g = grp[grp["condition"] == "shuffle"]
        assert len(g) == len(SHUFFLE_IDS)
        seed_rows.append({
            "battery_id": battery_id,
            "battery": battery,
            "seed": seed,
            "condition_summary": "shuffle_mean",
            "hits1": g["hits1"].mean(),
            "hits3": g["hits3"].mean(),
            "mrr": g["mrr"].mean(),
            "parameter_count": g["parameter_count"].mean(),
            "realization_sd_hits1": g["hits1"].std(),
        })

    g = grp[grp["condition"] == "rewire"]
    assert len(g) == len(REWIRE_IDS)
    seed_rows.append({
        "battery_id": battery_id,
        "battery": battery,
        "seed": seed,
        "condition_summary": "rewire_mean",
        "hits1": g["hits1"].mean(),
        "hits3": g["hits3"].mean(),
        "mrr": g["mrr"].mean(),
        "parameter_count": g["parameter_count"].mean(),
        "realization_sd_hits1": g["hits1"].std(),
    })

seed_summary = pd.DataFrame(seed_rows)

fold_mean = (
    seed_summary
    .groupby(["battery_id", "battery", "condition_summary"], as_index=False)
    .agg(
        hits1=("hits1", "mean"),
        hits3=("hits3", "mean"),
        mrr=("mrr", "mean"),
        parameter_count=("parameter_count", "mean"),
        realization_sd_hits1=("realization_sd_hits1", "mean"),
    )
)

aggregate = (
    fold_mean
    .groupby("condition_summary", as_index=False)
    .agg(
        hits1=("hits1", "mean"),
        hits3=("hits3", "mean"),
        mrr=("mrr", "mean"),
        mean_parameter_count=("parameter_count", "mean"),
        mean_realization_sd_hits1=("realization_sd_hits1", "mean"),
    )
)

display(aggregate.sort_values("hits1", ascending=False))


## 18. Exact product-level E2 contrasts

The six predeclared Hits@1 mechanism contrasts are:

1. MeanPool − NoEdge
2. Complete − NoEdge
3. Rewired − NoEdge
4. Real − MeanPool
5. Real − Complete
6. Real − Rewired

The first three ask whether progressively richer nonphysical context helps. The last three ask whether recorded physical locality adds value beyond those controls.

Holm-adjusted p-values across these six Hits@1 contrasts are reported as a diagnostic multiplicity correction. Hits@3 and MRR are supporting endpoints.


In [ ]:
# 18. E2 paired exact tests

def exact_two_sided_sign_flip(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    observed = abs(values.mean())
    extreme = 0
    total = 2 ** len(values)

    for signs in itertools.product((-1.0, 1.0), repeat=len(values)):
        stat = abs(np.mean(values * np.asarray(signs)))
        if stat >= observed - 1e-15:
            extreme += 1
    return extreme / total


def bootstrap_mean_ci(values, n_boot=20000, seed=20270220):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    draws = rng.choice(
        values,
        size=(n_boot, len(values)),
        replace=True,
    ).mean(axis=1)
    return (
        float(np.percentile(draws, 2.5)),
        float(np.percentile(draws, 97.5)),
    )


def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    m = len(pvalues)
    order = np.argsort(pvalues)
    adjusted_sorted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * pvalues[idx])
        adjusted_sorted[rank] = min(1.0, running)
    out = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        out[idx] = adjusted_sorted[rank]
    return out


wide = fold_mean.pivot(
    index=["battery_id", "battery"],
    columns="condition_summary",
    values=["hits1", "hits3", "mrr"],
)

contrast_specs = [
    ("MeanPool - NoEdge", "meanpool", "noedge"),
    ("Complete - NoEdge", "complete", "noedge"),
    ("Rewired - NoEdge", "rewire_mean", "noedge"),
    ("Real - MeanPool", "real", "meanpool"),
    ("Real - Complete", "real", "complete"),
    ("Real - Rewired", "real", "rewire_mean"),
]

if RUN_ORIGINAL_PERMUTATIONS:
    contrast_specs += [
        ("Permuted - MeanPool", "shuffle_mean", "meanpool"),
        ("Permuted - Complete", "shuffle_mean", "complete"),
        ("Permuted - Rewired", "shuffle_mean", "rewire_mean"),
    ]

rows = []
for metric_idx, metric in enumerate(["hits1", "hits3", "mrr"]):
    for j, (label, left, right) in enumerate(contrast_specs):
        d = (
            wide[(metric, left)]
            - wide[(metric, right)]
        ).to_numpy(float)

        ci = bootstrap_mean_ci(
            d,
            n_boot=20000,
            seed=20270220 + 100 * metric_idx + j,
        )
        rows.append({
            "metric": metric,
            "contrast": label,
            "mean_difference": float(d.mean()),
            "ci_low": ci[0],
            "ci_high": ci[1],
            "positive_products": int((d > 0).sum()),
            "negative_products": int((d < 0).sum()),
            "zero_products": int((d == 0).sum()),
            "p_exact": float(exact_two_sided_sign_flip(d)),
        })

contrast_stats = pd.DataFrame(rows)
contrast_stats["p_holm_primary6_hits1"] = np.nan

primary6 = set([
    "MeanPool - NoEdge",
    "Complete - NoEdge",
    "Rewired - NoEdge",
    "Real - MeanPool",
    "Real - Complete",
    "Real - Rewired",
])

mask = (
    (contrast_stats["metric"] == "hits1")
    & contrast_stats["contrast"].isin(primary6)
)

contrast_stats.loc[mask, "p_holm_primary6_hits1"] = holm_adjust(
    contrast_stats.loc[mask, "p_exact"].to_numpy(float)
)

display(
    contrast_stats[
        contrast_stats["metric"] == "hits1"
    ].sort_values("contrast")
)

print("\nSupporting Hits@3/MRR contrasts:")
display(
    contrast_stats[
        contrast_stats["metric"].isin(["hits3", "mrr"])
    ]
)


## 19. Rewiring diagnostics

Before interpreting the rewiring condition, verify that the control actually changed a substantial fraction of fixation records and inspect whether the number of connected components changed.


In [ ]:
# 19. Rewiring quality diagnostics

rewire_diag = (
    e2_results[e2_results["condition"] == "rewire"]
    .groupby(
        ["battery_id", "battery", "realization_id"],
        as_index=False,
    )
    .agg(
        changed_fraction=("rewire_changed_fraction", "first"),
        original_components=("rewire_original_components", "first"),
        rewired_components=("rewire_components", "first"),
    )
)

rewire_diag["component_count_preserved"] = (
    rewire_diag["original_components"]
    == rewire_diag["rewired_components"]
)

display(rewire_diag)

print(
    "Mean changed fixation-record fraction:",
    rewire_diag["changed_fraction"].mean(),
)
print(
    "Component-count preserved in",
    int(rewire_diag["component_count_preserved"].sum()),
    "of",
    len(rewire_diag),
    "battery-realization graphs.",
)


## 20. Save E2 results

Upload the generated `e2_global_context_results.zip` back to ChatGPT. The most important files are:

- `e2_all_seed_results.csv`
- `e2_seed_summary.csv`
- `e2_fold_mean.csv`
- `e2_aggregate.csv`
- `e2_contrasts.csv`
- `e2_rewire_diagnostics.csv`
- `e2_metadata.json`


In [ ]:
# 20. Save outputs and build lightweight ZIP

e2_results.to_csv(
    OUT / "e2_all_seed_results.csv",
    index=False,
)
seed_summary.to_csv(
    OUT / "e2_seed_summary.csv",
    index=False,
)
fold_mean.to_csv(
    OUT / "e2_fold_mean.csv",
    index=False,
)
aggregate.to_csv(
    OUT / "e2_aggregate.csv",
    index=False,
)
contrast_stats.to_csv(
    OUT / "e2_contrasts.csv",
    index=False,
)
rewire_diag.to_csv(
    OUT / "e2_rewire_diagnostics.csv",
    index=False,
)

metadata = {
    "experiment": "E2 global-context and structure-preserving controls",
    "independent_unit": "held-out battery product (n=10)",
    "seeds": [int(x) for x in SEEDS],
    "shuffle_ids": [int(x) for x in SHUFFLE_IDS],
    "rewire_ids": [int(x) for x in REWIRE_IDS],
    "run_original_permutations": bool(RUN_ORIGINAL_PERMUTATIONS),
    "epochs": int(EPOCHS),
    "negative_samples_per_state": int(NEGATIVES_PER_STATE),
    "learning_rate": float(LR),
    "weight_decay": float(WEIGHT_DECAY),
    "bootstrap_replicates": 20000,
    "primary_endpoint": "Hits@1",
    "notes": (
        "MeanPool uses no adjacency and zero graph-branch degree. "
        "CompleteGraph uses all-to-all edges with a single dummy edge type. "
        "Class-preserving rewiring preserves node degree, fixation edge type, "
        "and endpoint component-class pairs while changing exact adjacency."
    ),
}

with open(OUT / "e2_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

LIGHT = Path("/content/e2_global_context_results_light")
if LIGHT.exists():
    shutil.rmtree(LIGHT)
LIGHT.mkdir(parents=True, exist_ok=True)

for fn in [
    "e2_all_seed_results.csv",
    "e2_seed_summary.csv",
    "e2_fold_mean.csv",
    "e2_aggregate.csv",
    "e2_contrasts.csv",
    "e2_rewire_diagnostics.csv",
    "e2_metadata.json",
]:
    shutil.copy2(OUT / fn, LIGHT / fn)

zip_path = shutil.make_archive(
    "/content/e2_global_context_results",
    "zip",
    root_dir=LIGHT,
)

print("Saved:", zip_path)
for p in sorted(LIGHT.iterdir()):
    print(" -", p.name)
